# Notebook 1: Audio I/O and Preprocessing

**What:** How Demucs loads, resamples, and segments audio.

**Why:** Training and inference need consistent sample rate (44.1kHz), channel handling, and segment extraction.

**How:** Use `demucs.audio`, `torchaudio`, and understand segment logic.

## 1. Load Audio with torchaudio

Demucs expects: `(channels, samples)` at 44.1 kHz stereo.

In [ ]:
import sys
sys.path.insert(0, r'D:\demucs')

import torch
import torchaudio
from pathlib import Path

# Create a short test file if you don't have one
test_path = Path("test_audio.wav")
if not test_path.exists():
    # Generate 2 seconds of stereo noise
    sr = 44100
    audio = torch.randn(2, sr * 2) * 0.3
    torchaudio.save(str(test_path), audio, sr)
    print("Created test_audio.wav")

wav, sr = torchaudio.load(str(test_path))
print(f"Shape: {wav.shape} (channels, samples)")
print(f"Sample rate: {sr}")
print(f"Duration: {wav.shape[1] / sr:.2f} s")

## 2. Resampling and Channel Conversion

Demucs uses `julius` (or `torchaudio.transforms.Resample`) for resampling. Target: 44100 Hz, stereo.

In [ ]:
from demucs.audio import convert_audio_channels
import julius

def resample_if_needed(wav, sr, target_sr=44100):
    if sr != target_sr:
        wav = julius.resample_frac(wav, sr, target_sr)
    return wav

def to_stereo(wav):
    """Convert to 2 channels if mono."""
    return convert_audio_channels(wav, 2)

wav_processed = resample_if_needed(wav, sr)
wav_processed = to_stereo(wav_processed)
print(f"After processing: {wav_processed.shape}, 44100 Hz stereo")

## 3. Segment Extraction (Training Logic)

Training samples a random segment of length `segment * samplerate` from each track. This is how `wav.py` / dataset works conceptually.

In [ ]:
def extract_segment(wav, segment_samples, random_start=True):
    """
    Extract a segment from audio.
    wav: (C, T)
    segment_samples: length in samples
    """
    C, T = wav.shape
    if T <= segment_samples:
        return wav
    if random_start:
        start = torch.randint(0, T - segment_samples + 1, (1,)).item()
    else:
        start = 0
    return wav[:, start:start + segment_samples]

segment_sec = 4
sr = 44100
segment = extract_segment(wav_processed, segment_sec * sr, random_start=False)
print(f"Segment: {segment.shape} = {segment.shape[1]/sr:.2f} seconds")

## 4. Demucs AudioFile (for arbitrary formats)

For MP3, FLAC, etc., Demucs uses `demucs.audio.AudioFile` + ffmpeg. Requires ffmpeg on PATH.

In [ ]:
try:
    from demucs.audio import AudioFile
    af = AudioFile(test_path)
    print(f"AudioFile: {af}")
    data = af.read(samplerate=44100, channels=2)
    print(f"Read shape: {data.shape}")
except Exception as e:
    print(f"AudioFile needs ffmpeg: {e}")

## 5. Memory Estimate for 3070 Ti

For a segment of S seconds at 44.1kHz stereo:
- Samples per channel: 44100 * S
- Shape: (2, 44100*S)
- Float32: 4 bytes per sample → ~0.35 MB per second of stereo.

Segment 8s × batch 8 ≈ 2.8 MB input. Model weights + activations dominate; use small batches.

In [ ]:
segment_sec = 8
batch = 8
channels = 2
samples = 44100 * segment_sec
bytes_per_sample = 4
mb = (channels * samples * batch * bytes_per_sample) / 1e6
print(f"Batch input (8s, batch=8): ~{mb:.1f} MB")

**Next:** Notebook 2 — Model architectures (Demucs, HDemucs, HTDemucs).